## 1. Imports

In [1]:
!pip -q install -U openai
!pip install -U openai-agents python-dotenv
import os
import json
from dataclasses import dataclass
import numpy as np
import pandas as pd
import torch.optim as optim
import getpass
import copy
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from openai import OpenAI
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.8/470.8 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 19.9 MB/s eta 0:00:00


## 2. API key setup

Set your API key in the notebook session or in your environment before running the agent cells.

In [5]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

client = OpenAI()
print("OpenAI client is ready.")

OpenAI client is ready.


## 3. Load Stock Dataset

In [3]:
ds = load_dataset("Adilbai/stock-dataset")
df = ds["train"].to_pandas()

print(df.shape)
print(df.columns.tolist()[:20])
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


(620095, 73)
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal', 'MACD_Histogram', 'RSI', 'BB_Middle']


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,SMA_5,...,Future_Category_1d,Future_Return_5d,Future_Up_5d,Future_Category_5d,Future_Return_10d,Future_Up_10d,Future_Category_10d,Future_Return_20d,Future_Up_20d,Future_Category_20d
0,2020-09-08 00:00:00-04:00,93.113965,93.462744,91.806050,92.174202,1225600,0.0,0.0,A,96.138635,...,3.0,0.054972,1,3.0,0.036578,1,3.0,0.070141,1,3.0
1,2020-09-09 00:00:00-04:00,93.104280,95.400403,92.871764,94.877235,954400,0.0,0.0,A,95.520526,...,2.0,0.020321,1,3.0,-0.010416,0,1.0,0.062670,1,3.0
2,2020-09-10 00:00:00-04:00,95.797603,96.979573,95.177554,95.497269,1933200,0.0,0.0,A,94.580765,...,2.0,0.010449,1,2.0,-0.022623,0,0.0,0.058616,1,3.0
3,2020-09-11 00:00:00-04:00,95.632924,96.514560,94.964430,95.526352,1368600,0.0,0.0,A,94.379250,...,2.0,0.007911,1,2.0,-0.009229,0,1.0,0.074550,1,3.0
4,2020-09-14 00:00:00-04:00,96.621112,97.105525,95.884801,96.320770,1207700,0.0,0.0,A,94.879166,...,2.0,-0.012975,0,1.0,0.000905,1,2.0,0.062363,1,3.0


## 4. Feature Logic From GRU/LSTM Models

In [4]:
TARGET_COL = "Future_Up_1d"

leakage_cols = [
    "Future_Return_1d", "Future_Up_1d", "Future_Category_1d",
    "Future_Return_5d", "Future_Up_5d", "Future_Category_5d",
    "Future_Return_10d", "Future_Up_10d", "Future_Category_10d",
    "Future_Return_20d", "Future_Up_20d", "Future_Category_20d",
]

id_cols = ["Date", "Ticker"]

feature_cols = []
for c in df.columns:
    if c in leakage_cols or c in id_cols:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        feature_cols.append(c)

df = df.dropna(subset=feature_cols + [TARGET_COL, "Ticker", "Date"]).copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Rows:", len(df))
print("Features:", len(feature_cols))
print("Unique tickers:", df["Ticker"].nunique())

/tmp/ipykernel_22632/3891871958.py:20: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])


Rows: 620095
Features: 59
Unique tickers: 503


## 5. Create Time-Based Split

In [5]:
split_date = df["Date"].quantile(0.9)

train_df = df[df["Date"] <= split_date].copy()
val_df   = df[df["Date"] >  split_date].copy()

train_mean = train_df[feature_cols].mean()
train_std  = train_df[feature_cols].std().replace(0, 1.0)

train_df[feature_cols] = (train_df[feature_cols] - train_mean) / train_std
val_df[feature_cols]   = (val_df[feature_cols] - train_mean) / train_std

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Split date:", split_date)

Train rows: 558226
Val rows: 61869
Split date: 2024-12-27 00:00:00-05:00


## 6. Build Sequence Windows

In [6]:
SEQ_LEN = 20

def build_examples(data: pd.DataFrame, feature_cols, target_col, seq_len=20, max_per_ticker=None):
    examples = []

    for ticker, g in data.groupby("Ticker", sort=False):
        g = g.reset_index(drop=True)
        if len(g) < seq_len:
            continue

        counter = 0
        for end in range(seq_len - 1, len(g)):
            start = end - seq_len + 1
            window = g.loc[start:end].copy()
            row = g.loc[end].copy()

            examples.append({
                "Ticker": ticker,
                "Date": row["Date"],
                "target": int(row[target_col]),
                "window_df": window[["Date", "Ticker"] + feature_cols].copy()
            })

            counter += 1
            if max_per_ticker is not None and counter >= max_per_ticker:
                break

    return examples

# Keep this small first to control cost
train_examples = build_examples(train_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN, max_per_ticker=5)
val_examples   = build_examples(val_df,   feature_cols, TARGET_COL, seq_len=SEQ_LEN, max_per_ticker=2)

print("Train examples:", len(train_examples))
print("Val examples:", len(val_examples))
train_examples[0]["Ticker"], train_examples[0]["Date"], train_examples[0]["target"]

Train examples: 2515
Val examples: 1006


('A', Timestamp('2020-10-05 00:00:00-0400', tz='UTC-04:00'), 0)

## 7. Summarize each window into a compact prompt payload

This is where we make the stock context digestible for the LLM.

You can keep it very simple at first:
- last day feature snapshot
- mean / std over window
- simple trend deltas across the 20-day window

Later, you can add:
- company headlines
- earnings snippets
- macro news
- sector news

But start simple.

In [7]:
def summarize_window(window_df: pd.DataFrame, feature_cols, top_k_features=20):
    numeric = window_df[feature_cols].astype(float)

    # Keep only the first top_k_features for a cheap first prototype.
    use_cols = feature_cols[:top_k_features]
    numeric = numeric[use_cols]

    last_row = numeric.iloc[-1]
    mean_row = numeric.mean()
    std_row  = numeric.std().fillna(0.0)
    delta_row = numeric.iloc[-1] - numeric.iloc[0]

    summary = {
        "window_length": len(window_df),
        "last_date": str(window_df["Date"].iloc[-1].date()),
        "last_snapshot": {c: round(float(last_row[c]), 4) for c in use_cols},
        "window_mean":   {c: round(float(mean_row[c]), 4) for c in use_cols},
        "window_std":    {c: round(float(std_row[c]), 4) for c in use_cols},
        "window_delta":  {c: round(float(delta_row[c]), 4) for c in use_cols},
    }
    return summary

sample_summary = summarize_window(train_examples[0]["window_df"], feature_cols, top_k_features=12)
list(sample_summary.keys()), list(sample_summary["last_snapshot"].items())[:5]

(['window_length',
  'last_date',
  'last_snapshot',
  'window_mean',
  'window_std',
  'window_delta'],
 [('Open', -0.203),
  ('High', -0.1993),
  ('Low', -0.1994),
  ('Close', -0.1968),
  ('Volume', -0.2189)])

## 8. Optional place to add news later

Your professor mentioned current events. The cleanest progression is:

### Phase 1
Use only dataset-derived context.

### Phase 2
Add text news summaries for the same ticker and date range.

You can store them in a field called `news_context` and pass that to the agent. For now we start with a blank placeholder.

In [9]:
def get_news_context_stub(ticker: str, as_of_date: pd.Timestamp):
    # Replace this later with real headlines or summaries.
    # Keep blank for the first working version.
    return "No external news attached yet."

## 9. Define a structured output format

This keeps the LLM output easy to parse and train on.

In [10]:
@dataclass
class StockLLMLabel:
    ticker: str
    as_of_date: str
    prediction: str          # "UP" or "DOWN"
    confidence: float        # 0.0 to 1.0
    short_reason: str

## 10. Define the agent tool

This is the small "agentic" part:
the agent can call a tool to fetch stock context for an example id.

An agent in the OpenAI Agents SDK is configured with instructions and optional tools, and the docs recommend using the Responses-based model path. citeturn542339search1turn542339search7

In [11]:
# Store examples in a global dict so the tool can access them
example_store = {}

def refresh_example_store(examples, top_k_features=20):
    global example_store
    example_store = {}
    for i, ex in enumerate(examples):
        example_store[i] = {
            "ticker": ex["Ticker"],
            "as_of_date": str(pd.Timestamp(ex["Date"]).date()),
            "summary": summarize_window(ex["window_df"], feature_cols, top_k_features=top_k_features),
            "news_context": get_news_context_stub(ex["Ticker"], ex["Date"]),
        }

refresh_example_store(train_examples, top_k_features=20)
len(example_store)

2515

In [12]:
@function_tool
def get_stock_context(example_id: int) -> str:
    """Return stock context for a given example id as JSON."""
    if example_id not in example_store:
        return json.dumps({"error": f"example_id {example_id} not found"})
    return json.dumps(example_store[example_id], indent=2)

## 11. Create the agent

This agent has one job:
read stock context and return a direction label for the next move.

Keep the instructions narrow and explicit.

In [13]:
stock_agent = Agent(
    name="Stock Direction Research Agent",
    instructions=(
        "You are labeling stock examples for a machine learning dataset. "
        "Use the get_stock_context tool before answering. "
        "You must only use information available as of the provided date. "
        "Predict whether the stock is more likely to move UP or DOWN next. "
        "Return strict JSON with keys: ticker, as_of_date, prediction, confidence, short_reason. "
        "prediction must be either UP or DOWN. "
        "confidence must be a float between 0 and 1. "
        "Keep short_reason under 40 words."
    ),
    tools=[get_stock_context],
    model="gpt-4.1-mini"
)

## 12. Test the agent on one example

The SDK supports running an agent through `Runner.run_sync()` for synchronous execution. citeturn542339search11

In [14]:
test_prompt = (
    "Label example_id=0. "
    "Call the tool first, then output strict JSON only."
)

result = await Runner.run(stock_agent, test_prompt)
print(result.final_output)

{
  "ticker": "A",
  "as_of_date": "2020-10-05",
  "prediction": "UP",
  "confidence": 0.75,
  "short_reason": "MACD histogram rising sharply, RSI up strongly, and price above SMA indicating bullish momentum."
}


## 13. Parse the output safely

In [15]:
def parse_agent_json(raw_text: str):
    raw_text = raw_text.strip()

    # If the model wrapped JSON in markdown fences, strip them
    raw_text = raw_text.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    data = json.loads(raw_text)
    return StockLLMLabel(
        ticker=data["ticker"],
        as_of_date=data["as_of_date"],
        prediction=data["prediction"],
        confidence=float(data["confidence"]),
        short_reason=data["short_reason"],
    )

parsed = parse_agent_json(result.final_output)
parsed

StockLLMLabel(ticker='A', as_of_date='2020-10-05', prediction='UP', confidence=0.75, short_reason='MACD histogram rising sharply, RSI up strongly, and price above SMA indicating bullish momentum.')

## 14. Label a small batch first

Do **not** label the whole dataset first.
Start with a tiny subset to make sure:
- prompts are good
- JSON parsing works
- costs are reasonable
- answers are not nonsense

In [18]:
async def label_examples_with_agent(examples, limit, top_k_features):
    refresh_example_store(examples, top_k_features=top_k_features)

    rows = []
    for i in range(min(limit, len(examples))):
        prompt = f"Label example_id={i}. Call the tool first. Output strict JSON only."
        try:
            result = await Runner.run(stock_agent, prompt)
            parsed = parse_agent_json(result.final_output)

            rows.append({
                "example_id": i,
                "Ticker": parsed.ticker,
                "Date": parsed.as_of_date,
                "llm_prediction": parsed.prediction,
                "llm_confidence": parsed.confidence,
                "llm_short_reason": parsed.short_reason,
                "true_target": int(examples[i]["target"]),
            })
        except Exception as e:
            rows.append({
                "example_id": i,
                "Ticker": examples[i]["Ticker"],
                "Date": str(pd.Timestamp(examples[i]["Date"]).date()),
                "llm_prediction": None,
                "llm_confidence": None,
                "llm_short_reason": f"ERROR: {e}",
                "true_target": int(examples[i]["target"]),
            })

    return pd.DataFrame(rows)

LLM_TRAIN_LIMIT = min(100, len(train_examples))
LLM_VAL_LIMIT = min(100, len(val_examples))

llm_train_labels = await label_examples_with_agent(train_examples, limit=LLM_TRAIN_LIMIT, top_k_features=20)
llm_val_labels = await label_examples_with_agent(val_examples, limit=LLM_VAL_LIMIT, top_k_features=20)

print("Train LLM labels:", llm_train_labels.shape)
print("Val LLM labels:", llm_val_labels.shape)
llm_train_labels.head()


Train LLM labels: (100, 7)
Val LLM labels: (100, 7)


,example_id,Ticker,Date,llm_prediction,llm_confidence,llm_short_reason,true_target
0,0,A,2020-10-05,UP,0.75,MACD histogram positive and rising; RSI improv...,0
1,1,A,2020-10-06,UP,0.75,Positive MACD histogram surge and RSI rebound ...,1
2,2,A,2020-10-07,UP,0.80,Strong positive MACD histogram and RSI trendin...,1
3,3,A,2020-10-08,UP,0.80,Strong MACD histogram and rising RSI indicate ...,1
4,4,A,2020-10-09,UP,0.75,Positive MACD histogram and RSI suggest upward...,0


## 15. Quick evaluation of the raw LLM labels

Map:
- UP -> 1
- DOWN -> 0

In [19]:

eval_df = llm_train_labels.dropna(subset=["llm_prediction"]).copy()
eval_df["llm_pred_num"] = eval_df["llm_prediction"].map({"UP": 1, "DOWN": 0})

acc = accuracy_score(eval_df["true_target"], eval_df["llm_pred_num"])
print("LLM label accuracy on this batch:", round(acc, 4))
print(eval_df[["Ticker", "Date", "llm_prediction", "llm_confidence", "true_target"]].head(10))

LLM label accuracy on this batch: 0.47
  Ticker        Date llm_prediction  llm_confidence  true_target
0      A  2020-10-05             UP            0.75            0
1      A  2020-10-06             UP            0.75            1
2      A  2020-10-07             UP            0.80            1
3      A  2020-10-08             UP            0.80            1
4      A  2020-10-09             UP            0.75            0
5   AAPL  2020-08-11             UP            0.75            1
6   AAPL  2020-08-12             UP            0.75            1
7   AAPL  2020-08-13             UP            0.75            0
8   AAPL  2020-08-14             UP            0.75            0
9   AAPL  2020-08-17             UP            0.75            1


## 16. Train a simple student model on LLM outputs

There are two sensible first options.

### Option A
Treat the LLM as a teacher and train a simple student on numeric features.

### Option B
Use the LLM prediction/confidence as extra features inside your GRU/LSTM later.

For now, do Option A because it is much easier to debug.

In [20]:

def flatten_summary_to_features(summary_dict):
    feats = {}
    for block_name in ["last_snapshot", "window_mean", "window_std", "window_delta"]:
        for k, v in summary_dict[block_name].items():
            feats[f"{block_name}__{k}"] = float(v)
    return feats

def build_student_frame(examples, llm_labels_df, top_k_features=20):
    rows = []
    label_map = llm_labels_df.set_index("example_id").to_dict("index")

    for i, ex in enumerate(examples):
        if i not in label_map:
            continue
        row = label_map[i]
        if row["llm_prediction"] not in ["UP", "DOWN"]:
            continue

        summary = summarize_window(ex["window_df"], feature_cols, top_k_features=top_k_features)
        feats = flatten_summary_to_features(summary)

        feats["llm_pred_num"] = 1 if row["llm_prediction"] == "UP" else 0
        feats["llm_confidence"] = float(row["llm_confidence"])
        feats["true_target"] = int(ex["target"])
        feats["Ticker"] = ex["Ticker"]
        feats["Date"] = str(pd.Timestamp(ex["Date"]).date())
        rows.append(feats)

    return pd.DataFrame(rows)

student_df = build_student_frame(train_examples, llm_train_labels, top_k_features=20)
student_df.head()

,last_snapshot__Open,last_snapshot__High,last_snapshot__Low,last_snapshot__Close,last_snapshot__Volume,last_snapshot__Dividends,last_snapshot__Stock Splits,last_snapshot__SMA_5,last_snapshot__SMA_10,last_snapshot__SMA_20,...,window_delta__MACD_Histogram,window_delta__RSI,window_delta__BB_Middle,window_delta__BB_Upper,window_delta__BB_Lower,llm_pred_num,llm_confidence,true_target,Ticker,Date
0,-0.2030,-0.1993,-0.1994,-0.1968,-0.2189,1.1221,-0.0058,-0.2029,-0.2074,-0.2091,...,0.3619,1.1012,0.0017,0.0016,0.0017,1,0.75,0,A,2020-10-05
1,-0.1968,-0.1997,-0.1979,-0.2011,-0.2075,-0.0543,-0.0058,-0.2019,-0.2065,-0.2081,...,0.3578,0.3681,0.0024,0.0019,0.0030,1,0.75,1,A,2020-10-06
2,-0.1987,-0.1956,-0.1951,-0.1946,-0.1973,-0.0543,-0.0058,-0.2001,-0.2044,-0.2072,...,0.3868,0.6411,0.0032,0.0041,0.0022,1,0.80,1,A,2020-10-07
3,-0.1937,-0.1966,-0.1931,-0.1938,-0.2205,-0.0543,-0.0058,-0.1983,-0.2021,-0.2064,...,0.3929,0.7973,0.0040,0.0062,0.0016,1,0.80,1,A,2020-10-08
4,-0.1918,-0.1917,-0.1889,-0.1891,-0.2229,-0.0543,-0.0058,-0.1948,-0.1997,-0.2053,...,0.3997,1.0191,0.0048,0.0089,0.0002,1,0.75,0,A,2020-10-09


In [21]:

feature_drop = ["Ticker", "Date", "true_target"]
X = student_df.drop(columns=feature_drop)
y = student_df["true_target"].astype(int)

# Very small demo split because this is just a starter notebook
if len(student_df) >= 10:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    print("Student model accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print(classification_report(y_test, y_pred, zero_division=0))
else:
    print("Need more labeled examples before training the student model.")

Student model accuracy: 0.5333
              precision    recall  f1-score   support

           0       0.50      0.50      0.50        14
           1       0.56      0.56      0.56        16

    accuracy                           0.53        30
   macro avg       0.53      0.53      0.53        30
weighted avg       0.53      0.53      0.53        30



## 17. Save the LLM labels

This is useful because API calls cost money and time.
Save them once and reuse them.

In [22]:

os.makedirs("artifacts", exist_ok=True)

llm_train_labels.to_csv("artifacts/llm_train_labels.csv", index=False)
student_df.to_csv("artifacts/student_training_frame.csv", index=False)

print("Saved:")
print("- artifacts/llm_train_labels.csv")
print("- artifacts/student_training_frame.csv")

Saved:
- artifacts/llm_train_labels.csv
- artifacts/student_training_frame.csv


## GRU/LSTM probabilities

In [23]:
class StockSequenceDataset(Dataset):
    def __init__(self, data: pd.DataFrame, feature_cols, target_col, seq_len=20):
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.seq_len = seq_len

        self.groups = []
        for ticker, g in data.groupby("Ticker", sort=False):
            g = g.reset_index(drop=True)
            if len(g) >= seq_len:
                self.groups.append(g)

        self.indices = []
        for gi, g in enumerate(self.groups):
            for end in range(seq_len - 1, len(g)):
                self.indices.append((gi, end))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        gi, end = self.indices[idx]
        g = self.groups[gi]

        start = end - self.seq_len + 1
        x = g.loc[start:end, self.feature_cols].to_numpy(dtype=np.float32)
        y = np.float32(g.loc[end, self.target_col])

        return torch.from_numpy(x), torch.tensor(y)

BATCH_SIZE = 256

train_dataset = StockSequenceDataset(train_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN)
val_dataset = StockSequenceDataset(val_df, feature_cols, TARGET_COL, seq_len=SEQ_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train sequences:", len(train_dataset))
print("Val sequences:", len(val_dataset))

class GRUClassifier(nn.Module):
    def __init__(self, num_features, hidden_dim=64):
        super().__init__()
        self.gru = nn.GRU(
            input_size=num_features,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        last = out[:, -1, :]
        return self.fc(last).squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUClassifier(num_features=len(feature_cols)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)


Train sequences: 548669
Val sequences: 52312


In [24]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device).float()

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / len(loader.dataset)
    acc = correct / total
    return avg_loss, acc


@torch.no_grad()
def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device).float()

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / len(loader.dataset)
    acc = correct / total
    return avg_loss, acc


best_val_loss = float("inf")
patience = 3
counter = 0
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_model_state)

Epoch 01 | Train Loss: 0.6919 | Train Acc: 0.5200 | Val Loss: 0.6923 | Val Acc: 0.5174
Epoch 02 | Train Loss: 0.6908 | Train Acc: 0.5263 | Val Loss: 0.6909 | Val Acc: 0.5181
Epoch 03 | Train Loss: 0.6896 | Train Acc: 0.5323 | Val Loss: 0.6925 | Val Acc: 0.5206
Epoch 04 | Train Loss: 0.6883 | Train Acc: 0.5367 | Val Loss: 0.6926 | Val Acc: 0.5188
Epoch 05 | Train Loss: 0.6870 | Train Acc: 0.5410 | Val Loss: 0.6939 | Val Acc: 0.5158
Early stopping triggered.


<All keys matched successfully>

In [25]:
def get_sequence_model_predictions(model, examples, feature_cols, device):
    model.eval()
    rows = []

    with torch.no_grad():
        for i, ex in enumerate(examples):
            x = ex["window_df"][feature_cols].astype(float).values
            x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # (1, seq_len, num_features)

            logit = model(x_tensor)
            prob = torch.sigmoid(logit).item()

            rows.append({
                "example_id": i,
                "Ticker": ex["Ticker"],
                "Date": str(pd.Timestamp(ex["Date"]).date()),
                "gru_prob": prob,
                "true_target": int(ex["target"]),
            })

    return pd.DataFrame(rows)
gru_train_preds = get_sequence_model_predictions(model, train_examples, feature_cols, device)
gru_val_preds = get_sequence_model_predictions(model, val_examples, feature_cols, device)

## Prepare LLM outputs

In [26]:
def prep_llm_labels(llm_df):
    out = llm_df.copy()
    out = out[out["llm_prediction"].isin(["UP", "DOWN"])].copy()
    out["llm_pred_num"] = out["llm_prediction"].map({"UP": 1, "DOWN": 0})
    out["llm_confidence"] = pd.to_numeric(out["llm_confidence"], errors="coerce")
    out = out.dropna(subset=["llm_pred_num", "llm_confidence"])
    out["llm_confidence"] = out["llm_confidence"].clip(0.0, 1.0)
    return out[["example_id", "llm_pred_num", "llm_confidence"]]

llm_train_prepped = prep_llm_labels(llm_train_labels)
llm_val_prepped = prep_llm_labels(llm_val_labels)


## Merge GRU/LSTM + LLM outputs

In [27]:
ensemble_train_df = gru_train_preds.merge(llm_train_prepped, on="example_id", how="inner")
ensemble_val_df = gru_val_preds.merge(llm_val_prepped, on="example_id", how="inner")

ensemble_train_df.head()
ensemble_val_df.head()

,example_id,Ticker,Date,gru_prob,true_target,llm_pred_num,llm_confidence
0,0,A,2025-01-29,0.556939,1,1,0.80
1,1,A,2025-01-30,0.537060,1,1,0.75
2,2,AAPL,2025-01-29,0.472548,0,0,0.85
3,3,AAPL,2025-01-30,0.466956,0,0,0.80
4,4,ABBV,2025-01-29,0.544259,1,1,0.70


## Train the final ensemble model

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

ensemble_features = ["gru_prob", "llm_pred_num", "llm_confidence"]

X_train_ens = ensemble_train_df[ensemble_features]
y_train_ens = ensemble_train_df["true_target"]

X_val_ens = ensemble_val_df[ensemble_features]
y_val_ens = ensemble_val_df["true_target"]

ensemble_model = LogisticRegression(max_iter=1000)
ensemble_model.fit(X_train_ens, y_train_ens)

val_pred = ensemble_model.predict(X_val_ens)
val_prob = ensemble_model.predict_proba(X_val_ens)[:, 1]

print("Ensemble Accuracy:", accuracy_score(y_val_ens, val_pred))
print("Ensemble ROC AUC:", roc_auc_score(y_val_ens, val_prob))
print(classification_report(y_val_ens, val_pred))

Ensemble Accuracy: 0.58
Ensemble ROC AUC: 0.6030844155844156
              precision    recall  f1-score   support

           0       0.52      0.70      0.60        44
           1       0.68      0.48      0.56        56

    accuracy                           0.58       100
   macro avg       0.60      0.59      0.58       100
weighted avg       0.61      0.58      0.58       100

